# Silver → Gold preprocessing (working notebook)

Loads the silver records **not yet in Redshift** into a pandas DataFrame
(reusing `etl/warehouse/transform.py`), then scaffolds per-column preprocessing
for you to extend. Run top-to-bottom once, then iterate on the processing cells.
The cells below cover **all 21 columns** — working examples + `TODO`s.

In [1]:
# --- setup: make the `etl` package importable + load .env ---
import os, sys, json
import pandas as pd

# Walk up from the notebook's cwd to the ETL root (the dir containing the `etl` package).
ETL_ROOT = os.getcwd()
while ETL_ROOT != "/" and not os.path.isdir(os.path.join(ETL_ROOT, "etl")):
    ETL_ROOT = os.path.dirname(ETL_ROOT)
if ETL_ROOT not in sys.path:
    sys.path.insert(0, ETL_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(ETL_ROOT, ".env"))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 40)
print("ETL_ROOT =", ETL_ROOT)

ETL_ROOT = /home/skynet/Downloads/Data engineering/ETL


In [2]:
# --- load the not-yet-loaded silver into a DataFrame (reuses transform.py) ---
# Hits Redshift + Postgres (anti-join) then fetches each silver JSON from S3 (~30-60s).
from etl.warehouse.transform import get_silver_df

df = get_silver_df()
print("shape:", df.shape)
df.head()

shape: (623, 21)


,message_id,account_id,sender,subject,email_date,vendor,category,currency,total_amount,transaction_id,line_items,summary,metadata,template_hash,confidence,extracted_at,extraction_status,_errors,silver_id,user_id,google_account_id
0,19ebab1d1dda6ea2,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Friday afternoon trip with Uber,"Fri, 12 Jun 2026 07:17:55 +0000",Uber,ride_hailing,INR,95.50,None,"[{'quantity': 1, 'description': 'Sug...",{'total': 95.5},{'Promotion': '-₹5.03'},50095ec69397386a9c71b1a6d6879aff6007...,0.8,2026-06-23T13:19:22.885128+00:00,extracted,[],bff95c65-0767-42ea-8cad-c1e0d7d164d6,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9
1,19eb176b91cb2785,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Wednesday evening trip with Uber,"Wed, 10 Jun 2026 12:16:48 +0000 (UTC)",Uber,ride_hailing,INR,93.56,None,"[{'quantity': 1, 'description': 'Sug...",{'total': 93.56},{},d5448b03738047fd54f7d1e0a381122323e5...,0.8,2026-06-23T13:19:23.448655+00:00,extracted,[],c07a0170-b52c-4c71-bd6b-273f4fc76d25,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9
2,19eb08f5bb7c9e87,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Wednesday afternoon trip with Uber,"Wed, 10 Jun 2026 08:04:04 +0000",Uber,ride_hailing,INR,79.04,None,"[{'quantity': 1, 'description': 'Sug...",{'total': 79.04},{},f4366ded0b48ec80e4a3f94efe52c6602ef0...,0.8,2026-06-23T13:19:23.924226+00:00,extracted,[],6aed86b2-adce-49f6-aa3b-3488845e33bc,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9
3,19e9cd6a268c7c5d,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Saturday evening trip with Uber,"Sat, 06 Jun 2026 12:09:32 +0000 (UTC)",Uber,ride_hailing,INR,45.13,None,"[{'quantity': 1, 'description': 'Sug...",{'total': 45.13},{},732e716d69606b3448b8936959d9f2e6ebf9...,0.8,2026-06-23T13:19:24.413074+00:00,extracted,[],91d7eb04-0e04-4f72-8ab0-404e7c5750d7,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9
4,19e9cc5e8e5de749,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Saturday evening trip with Uber,"Sat, 06 Jun 2026 11:51:16 +0000",Uber,ride_hailing,INR,93.85,None,"[{'quantity': 1, 'description': 'Sug...",{'total': 93.85},{},f34af094d87a4a4186fc79360df33ddabc2c...,0.8,2026-06-23T13:19:25.240337+00:00,extracted,[],e5c3236a-9a9e-4c70-adc0-99070f413570,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9


In [ ]:
# Optional: cache locally so you don't re-fetch from S3 on every kernel restart.
# df.to_parquet("/tmp/silver.parquet")
# df = pd.read_parquet("/tmp/silver.parquet")

In [3]:
# --- inspect every column: dtype, nulls, uniques, a sample value ---
def column_overview(d):
    rows = []
    for c in d.columns:
        nn = d[c].dropna()
        rows.append({
            "column":   c,
            "dtype":    str(d[c].dtype),
            "nulls":    int(d[c].isna().sum()),
            "n_unique": int(nn.apply(lambda v: json.dumps(v, default=str)).nunique()) if len(nn) else 0,
            "sample":   repr(nn.iloc[0])[:60] if len(nn) else None,
        })
    return pd.DataFrame(rows)

column_overview(df)

,column,dtype,nulls,n_unique,sample
0,message_id,str,0,623,'19ebab1d1dda6ea2'
1,account_id,str,0,1,'3b3abf43-ab93-4b01-a68b-2935a5f3d3f9'
2,sender,str,0,4,'Uber Receipts <noreply@uber.com>'
3,subject,str,0,74,'Your Friday afternoon trip with Uber'
4,email_date,str,0,623,"'Fri, 12 Jun 2026 07:17:55 +0000'"
5,vendor,str,0,1,'Uber'
6,category,str,0,2,'ride_hailing'
7,currency,str,0,1,'INR'
8,total_amount,float64,41,419,np.float64(95.5)
9,transaction_id,object,623,0,NaN


## Per-column preprocessing

We build a `clean` copy and transform it column-group by column-group. Each cell
has a working starting point; extend the `TODO`s as you go.

In [5]:
clean = df.copy()
print(clean.shape)

(623, 21)


### 1. Dates — `email_date`, `extracted_at`

In [6]:
# email_date: RFC-2822 string (e.g. 'Fri, 12 Jun 2026 07:17:55 +0000') -> datetime
clean["email_date"] = pd.to_datetime(clean["email_date"], errors="coerce", utc=True, format="mixed")
# derive the dim_date surrogate key (YYYYMMDD); NaT -> <NA>
clean["date_key"] = pd.to_numeric(clean["email_date"].dt.strftime("%Y%m%d"), errors="coerce").astype("Int64")

# extracted_at: ISO-8601 string -> datetime
clean["extracted_at"] = pd.to_datetime(clean["extracted_at"], errors="coerce", utc=True, format="mixed")

clean[["email_date", "date_key", "extracted_at"]].head()
# TODO: if you need receipt_date separately, parse it here too.

,email_date,date_key,extracted_at
0,2026-06-12 07:17:55+00:00,20260612,2026-06-23 13:19:22.885128+00:00
1,2026-06-10 12:16:48+00:00,20260610,2026-06-23 13:19:23.448655+00:00
2,2026-06-10 08:04:04+00:00,20260610,2026-06-23 13:19:23.924226+00:00
3,2026-06-06 12:09:32+00:00,20260606,2026-06-23 13:19:24.413074+00:00
4,2026-06-06 11:51:16+00:00,20260606,2026-06-23 13:19:25.240337+00:00


### 2. Numeric measures — `total_amount`, `confidence`

In [7]:
clean["total_amount"] = pd.to_numeric(clean["total_amount"], errors="coerce")
clean["confidence"]   = pd.to_numeric(clean["confidence"],   errors="coerce")
clean[["total_amount", "confidence"]].describe()
# TODO: add tax/amount here if the silver extractor ever yields them.

,total_amount,confidence
count,582.000000,623.000000
mean,58.985773,0.763884
std,33.124585,0.124476
min,0.000000,0.300000
25%,40.930000,0.800000
50%,59.020000,0.800000
75%,68.877500,0.800000
max,257.770000,0.800000


### 3. Categorical / text — `vendor`, `category`, `currency`, `extraction_status`, `sender`, `subject`

In [8]:
for c in ["vendor", "category", "currency", "extraction_status"]:
    print(c, "->", clean[c].value_counts(dropna=False).to_dict())

# TODO examples:
# clean["currency"] = clean["currency"].str.upper().fillna("INR")
# clean["vendor"]   = clean["vendor"].str.strip()

vendor -> {'Uber': 623}
category -> {'ride_hailing': 616, 'other': 7}
currency -> {'INR': 623}
extraction_status -> {'extracted': 582, 'quarantine': 41}


### 4. Nested / JSON columns — `line_items`, `summary`, `metadata`

In [9]:
# These are python list/dict objects in the df. The warehouse stores them as VARCHAR,
# so serialize to JSON strings:
for c in ["line_items", "summary", "metadata"]:
    clean[c] = clean[c].apply(lambda v: json.dumps(v, ensure_ascii=False) if v is not None else None)

clean[["line_items", "summary", "metadata"]].head()
# TODO (alt): explode line_items into a line-grain frame for a future fact_line_item table:
# items = df[["silver_id","line_items"]].explode("line_items")

,line_items,summary,metadata
0,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 95.5}","{""Promotion"": ""-₹5.03""}"
1,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 93.56}",{}
2,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 79.04}",{}
3,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 45.13}",{}
4,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 93.85}",{}


### 5. Filter / quarantine — split off bad extractions

In [10]:
quarantined = clean[clean["extraction_status"] == "quarantine"].copy()
clean = clean[clean["extraction_status"] != "quarantine"].copy()
print("kept:", len(clean), " quarantined:", len(quarantined))
# TODO: decide whether quarantined rows go to a side table or are just skipped.

kept: 582  quarantined: 41


### 6. IDs / passthrough — `message_id`, `transaction_id`, `silver_id`, `user_id`, `account_id`, `google_account_id`, `template_hash`, `_errors`

In [11]:
# Mostly passthrough. A couple of conventional renames toward fact_receipts:
clean = clean.rename(columns={
    "message_id": "email_id",            # fact grain
    "transaction_id": "invoice_number",  # degenerate dimension
})
# TODO: drop what you won't load to gold, e.g.:
# clean = clean.drop(columns=["_errors", "template_hash", "subject", "sender", "summary"])

### 7. Result — your gold-ready frame

In [12]:
print("clean shape:", clean.shape)
print("columns:", list(clean.columns))
clean.head()
# TODO: next step (separate notebook/job) -> resolve vendor_key from dim_vendor,
#       then load `clean` into Redshift fact_receipts via staging + MERGE.

clean shape: (582, 22)
columns: ['email_id', 'account_id', 'sender', 'subject', 'email_date', 'vendor', 'category', 'currency', 'total_amount', 'invoice_number', 'line_items', 'summary', 'metadata', 'template_hash', 'confidence', 'extracted_at', 'extraction_status', '_errors', 'silver_id', 'user_id', 'google_account_id', 'date_key']


,email_id,account_id,sender,subject,email_date,vendor,category,currency,total_amount,invoice_number,line_items,summary,metadata,template_hash,confidence,extracted_at,extraction_status,_errors,silver_id,user_id,google_account_id,date_key
0,19ebab1d1dda6ea2,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Friday afternoon trip with Uber,2026-06-12 07:17:55+00:00,Uber,ride_hailing,INR,95.50,None,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 95.5}","{""Promotion"": ""-₹5.03""}",50095ec69397386a9c71b1a6d6879aff6007...,0.8,2026-06-23 13:19:22.885128+00:00,extracted,[],bff95c65-0767-42ea-8cad-c1e0d7d164d6,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,20260612
1,19eb176b91cb2785,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Wednesday evening trip with Uber,2026-06-10 12:16:48+00:00,Uber,ride_hailing,INR,93.56,None,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 93.56}",{},d5448b03738047fd54f7d1e0a381122323e5...,0.8,2026-06-23 13:19:23.448655+00:00,extracted,[],c07a0170-b52c-4c71-bd6b-273f4fc76d25,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,20260610
2,19eb08f5bb7c9e87,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Wednesday afternoon trip with Uber,2026-06-10 08:04:04+00:00,Uber,ride_hailing,INR,79.04,None,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 79.04}",{},f4366ded0b48ec80e4a3f94efe52c6602ef0...,0.8,2026-06-23 13:19:23.924226+00:00,extracted,[],6aed86b2-adce-49f6-aa3b-3488845e33bc,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,20260610
3,19e9cd6a268c7c5d,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Saturday evening trip with Uber,2026-06-06 12:09:32+00:00,Uber,ride_hailing,INR,45.13,None,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 45.13}",{},732e716d69606b3448b8936959d9f2e6ebf9...,0.8,2026-06-23 13:19:24.413074+00:00,extracted,[],91d7eb04-0e04-4f72-8ab0-404e7c5750d7,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,20260606
4,19e9cc5e8e5de749,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,Uber Receipts <noreply@uber.com>,Your Saturday evening trip with Uber,2026-06-06 11:51:16+00:00,Uber,ride_hailing,INR,93.85,None,"[{""quantity"": 1, ""description"": ""Sug...","{""total"": 93.85}",{},f34af094d87a4a4186fc79360df33ddabc2c...,0.8,2026-06-23 13:19:25.240337+00:00,extracted,[],e5c3236a-9a9e-4c70-adc0-99070f413570,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,20260606


In [13]:
# get type of clean df columns
clean.dtypes

email_id                             str
account_id                           str
sender                               str
subject                              str
email_date           datetime64[us, UTC]
vendor                               str
category                             str
currency                             str
total_amount                     float64
invoice_number                    object
line_items                           str
summary                              str
metadata                             str
template_hash                        str
confidence                       float64
extracted_at         datetime64[us, UTC]
extraction_status                    str
_errors                           object
silver_id                            str
user_id                              str
google_account_id                    str
date_key                           Int64
dtype: object

In [ ]:
# get all the tables and columns in redshift and map them to the clean df columns. This will help in creating the fact table in redshift.

## Map to `fact_receipts` — schema-driven

The target column list is pulled from the `FactReceipts` SQLAlchemy model (the
single source of truth), **not hardcoded**. The per-field assignments below are
explicit (semantic renames like `total_amount`→`total` can't be inferred), but
the final column set + order come from the schema, and we validate coverage.

In [14]:
# fact columns straight from the model (no DB connection needed — engine is lazy)
from etl.warehouse.schema import FactReceipts

FACT_COLUMNS   = [c.name for c in FactReceipts.__table__.columns]
# loaded_at has a server default (GETDATE()), so we don't insert it
INSERT_COLUMNS = [c for c in FACT_COLUMNS if c != "loaded_at"]
print(len(INSERT_COLUMNS), "insert columns:")
print(INSERT_COLUMNS)

21 insert columns:
['email_id', 'user_id', 'vendor_key', 'date_key', 'account_id', 'currency', 'amount', 'tax', 'total', 'line_items', 'metadata', 'receipt_date', 'email_date', 'invoice_number', 'confidence_score', 'extraction_status', 'anomaly_flag', 'anomaly_z_score', 'deleted_at', 'silver_id', 'extracted_at']


In [15]:
import json
import pandas as pd

def to_fact_df(df, columns=INSERT_COLUMNS, drop_quarantine=True):
    """silver DataFrame (get_silver_df) -> fact_receipts-ready DataFrame.
    Target columns come from the schema (`columns`); per-field mapping is explicit."""
    d = df.copy()
    if drop_quarantine:
        d = d[d["extraction_status"] != "quarantine"].copy()

    email_dt     = pd.to_datetime(d["email_date"],   errors="coerce", utc=True, format="mixed")
    extracted_dt = pd.to_datetime(d["extracted_at"], errors="coerce", utc=True, format="mixed")

    def _dumps(v):
        return None if v is None else json.dumps(v, ensure_ascii=False)

    def _meta(r):
        m = dict(r["metadata"] or {})
        if r.get("summary"):
            m["_summary"] = r["summary"]
        return json.dumps(m, ensure_ascii=False)

    out = pd.DataFrame({
        "email_id":          d["message_id"],
        "user_id":           d["user_id"],
        "vendor_key":        pd.NA,                                  # resolve later via dim_vendor
        "date_key":          pd.to_numeric(email_dt.dt.strftime("%Y%m%d"), errors="coerce").astype("Int64"),
        "account_id":        d["account_id"].fillna(d["google_account_id"]),
        "currency":          d["currency"],
        "amount":            pd.NA,                                  # not extracted today
        "tax":               pd.NA,
        "total":             pd.to_numeric(d["total_amount"], errors="coerce"),
        "line_items":        d["line_items"].apply(_dumps),
        "metadata":          d.apply(_meta, axis=1),
        "receipt_date":      email_dt.dt.date,
        "email_date":        email_dt.dt.date,
        "invoice_number":    d["transaction_id"],
        "confidence_score":  pd.to_numeric(d["confidence"], errors="coerce"),
        "extraction_status": d["extraction_status"],
        "anomaly_flag":      False,
        "anomaly_z_score":   pd.NA,
        "deleted_at":        pd.NaT,
        "silver_id":         d["silver_id"],
        "extracted_at":      extracted_dt.dt.tz_localize(None),      # Redshift timestamp has no tz
    })

    # schema-driven: add any fact columns we didn't set (as NA), then order by schema
    for c in columns:
        if c not in out.columns:
            out[c] = pd.NA
    return out[columns].reset_index(drop=True)

In [16]:
fact_df = to_fact_df(df)

# validate the mapping covers the schema exactly
missing = set(INSERT_COLUMNS) - set(fact_df.columns)
extra   = set(fact_df.columns) - set(INSERT_COLUMNS)
print("fact_df:", fact_df.shape)
print("missing schema columns:", missing or "none")
print("extra columns:", extra or "none")
fact_df.head()

fact_df: (582, 21)
missing schema columns: none
extra columns: none


,email_id,user_id,vendor_key,date_key,account_id,currency,amount,tax,total,line_items,metadata,receipt_date,email_date,invoice_number,confidence_score,extraction_status,anomaly_flag,anomaly_z_score,deleted_at,silver_id,extracted_at
0,19ebab1d1dda6ea2,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,<NA>,20260612,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,INR,<NA>,<NA>,95.50,"[{""quantity"": 1, ""description"": ""Sug...","{""Promotion"": ""-₹5.03"", ""_summary"": ...",2026-06-12,2026-06-12,None,0.8,extracted,False,<NA>,NaT,bff95c65-0767-42ea-8cad-c1e0d7d164d6,2026-06-23 13:19:22.885128
1,19eb176b91cb2785,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,<NA>,20260610,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,INR,<NA>,<NA>,93.56,"[{""quantity"": 1, ""description"": ""Sug...","{""_summary"": {""total"": 93.56}}",2026-06-10,2026-06-10,None,0.8,extracted,False,<NA>,NaT,c07a0170-b52c-4c71-bd6b-273f4fc76d25,2026-06-23 13:19:23.448655
2,19eb08f5bb7c9e87,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,<NA>,20260610,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,INR,<NA>,<NA>,79.04,"[{""quantity"": 1, ""description"": ""Sug...","{""_summary"": {""total"": 79.04}}",2026-06-10,2026-06-10,None,0.8,extracted,False,<NA>,NaT,6aed86b2-adce-49f6-aa3b-3488845e33bc,2026-06-23 13:19:23.924226
3,19e9cd6a268c7c5d,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,<NA>,20260606,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,INR,<NA>,<NA>,45.13,"[{""quantity"": 1, ""description"": ""Sug...","{""_summary"": {""total"": 45.13}}",2026-06-06,2026-06-06,None,0.8,extracted,False,<NA>,NaT,91d7eb04-0e04-4f72-8ab0-404e7c5750d7,2026-06-23 13:19:24.413074
4,19e9cc5e8e5de749,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,<NA>,20260606,3b3abf43-ab93-4b01-a68b-2935a5f3d3f9,INR,<NA>,<NA>,93.85,"[{""quantity"": 1, ""description"": ""Sug...","{""_summary"": {""total"": 93.85}}",2026-06-06,2026-06-06,None,0.8,extracted,False,<NA>,NaT,e5c3236a-9a9e-4c70-adc0-99070f413570,2026-06-23 13:19:25.240337


## Refresh dimensions BEFORE the fact load

New dimension members (vendors, users) seen in this batch are inserted first, so
the fact load can resolve its FKs. Pattern = natural-key **anti-join + insert-new**
(SCD type 1). `date_key` is already covered by the seeded `dim_date`.

In [17]:
import uuid
from email.utils import parseaddr
from etl.warehouse.connection import get_conn
from etl.db import postgres

def sender_email_of(sender):
    """'Uber <noreply@uber.com>' -> 'noreply@uber.com' (lowercased)."""
    return (parseaddr(sender or "")[1] or "").lower() or None


def upsert_dim_vendor(df):
    """Insert vendors new to this batch; return {sender_email -> vendor_key} for ALL."""
    tmp = (df.assign(sender_email=df["sender"].map(sender_email_of))
             .dropna(subset=["sender_email"]).drop_duplicates("sender_email"))
    batch = {r["sender_email"]: (r.get("vendor"), r.get("category"), r.get("template_hash"))
             for _, r in tmp.iterrows()}

    conn = get_conn()
    try:
        cur = conn.cursor()
        cur.execute("SELECT sender_email, vendor_key FROM dim_vendor")
        keymap = {se: str(vk) for se, vk in cur.fetchall()}     # existing
        for se, (vname, cat, th) in batch.items():
            if se in keymap:
                continue                                        # already present (SCD-1: leave as-is)
            vk = str(uuid.uuid4())
            cur.execute(
                "INSERT INTO dim_vendor "
                "(vendor_key, sender_email, vendor_name, category, template_hash, is_active, synced_at) "
                "VALUES (%s, %s, %s, %s, %s, %s, GETDATE())",
                (vk, se, vname, cat, th, True),
            )
            keymap[se] = vk
        conn.commit()
    finally:
        conn.close()
    return keymap

In [18]:
def sync_dim_user(user_ids):
    """Insert dim_user rows new to this batch (sourced from Postgres users). user_key == users.id."""
    ids = [str(u) for u in set(user_ids) if u]
    if not ids:
        return set()
    ph = ",".join(["%s"] * len(ids))
    users = postgres._query_all(f"SELECT id, email, name FROM users WHERE id IN ({ph})", tuple(ids))
    accts = postgres._query_all(f"SELECT user_id, id FROM google_accounts WHERE user_id IN ({ph})", tuple(ids))
    acct_map = {}
    for uid, aid in accts:
        acct_map.setdefault(str(uid), []).append(str(aid))

    conn = get_conn()
    try:
        cur = conn.cursor()
        cur.execute("SELECT user_key FROM dim_user")
        existing = {str(r[0]) for r in cur.fetchall()}
        for uid, email, name in users:
            uid = str(uid)
            if uid in existing:
                continue
            cur.execute(
                "INSERT INTO dim_user (user_key, email, name, role, account_ids, is_active, synced_at) "
                "VALUES (%s, %s, %s, %s, %s, %s, GETDATE())",
                (uid, email, name, "user", json.dumps(acct_map.get(uid, [])), True),
            )
            existing.add(uid)
        conn.commit()
    finally:
        conn.close()
    return existing

### Build the fact frame with FKs resolved

In [19]:
# 1. refresh dims (insert any new members), get the vendor key map
vendor_map = upsert_dim_vendor(df)
sync_dim_user(df["user_id"].unique())

# 2. build the fact rows (vendor_key still <NA>)
fact_df = to_fact_df(df)

# 3. resolve vendor_key per receipt via its sender (join on email_id == message_id)
mid_to_vendor = {r["message_id"]: vendor_map.get(sender_email_of(r["sender"]))
                 for _, r in df.iterrows()}
fact_df["vendor_key"] = fact_df["email_id"].map(mid_to_vendor)

print("fact rows:", len(fact_df))
print("distinct vendors in dim:", len(vendor_map))
print("null vendor_key:", int(fact_df["vendor_key"].isna().sum()), "(should be 0)")
print("null user_id:", int(fact_df["user_id"].isna().sum()), "| null date_key:", int(fact_df["date_key"].isna().sum()))
fact_df[["email_id", "vendor_key", "user_id", "date_key", "currency", "total"]].head()

fact rows: 582
distinct vendors in dim: 1
null vendor_key: 0 (should be 0)
null user_id: 0 | null date_key: 0


,email_id,vendor_key,user_id,date_key,currency,total
0,19ebab1d1dda6ea2,235347ff-4a13-4a92-b74c-7c534cf74df9,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,20260612,INR,95.50
1,19eb176b91cb2785,235347ff-4a13-4a92-b74c-7c534cf74df9,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,20260610,INR,93.56
2,19eb08f5bb7c9e87,235347ff-4a13-4a92-b74c-7c534cf74df9,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,20260610,INR,79.04
3,19e9cd6a268c7c5d,235347ff-4a13-4a92-b74c-7c534cf74df9,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,20260606,INR,45.13
4,19e9cc5e8e5de749,235347ff-4a13-4a92-b74c-7c534cf74df9,eb1b8a6a-efbe-41c0-8e9f-3c7cdf37b90b,20260606,INR,93.85
